In [ ]:
# # Code to convert this notebook to .py if you want to run it via command line or with Slurm
# from subprocess import call
# command = "jupyter nbconvert Train_MindEye.ipynb --to python"
# call(command,shell=True)

In [ ]:
import os
import sys
import json
import argparse
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tqdm import tqdm
import torchvision.transforms as transforms
import transformers

# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

# custom models and functions #
import utils
from models import Clipper, OpenClipper, BrainNetwork, BrainDiffusionPrior, BrainDiffusionPriorOld, VersatileDiffusionPriorNetwork

from models import BrainNetwork, PriorNetwork, BrainDiffusionPriorV2

# Multi-GPU config #
from accelerate import Accelerator
accelerator = Accelerator(split_batches=False,mixed_precision='fp16')  
print("PID of this process =",os.getpid())
print = accelerator.print # only print if local_rank=0
device = accelerator.device
print("device:",device)
num_devices = torch.cuda.device_count()
if num_devices==0: num_devices = 1
num_workers = num_devices
print(accelerator.state)
local_rank = accelerator.state.local_process_index
world_size = accelerator.state.num_processes
distributed = not accelerator.state.distributed_type == 'NO'
print("distributed =",distributed, "num_devices =", num_devices, "local rank =", local_rank, "world size =", world_size)

In [ ]:
# if running this interactively, can specify jupyter_args here for argparser to use
if utils.is_interactive():
    brain2clip_jupyter_args = "--data_path=/brain-decoding/fmri/natural-scenes-dataset \
                    --model_name=prior_257x1280_subj02_bs32_heads20_no_img_aug_preproc_vith\
                    --subj=1 --hidden --no-norm_embs --clip_variant=ViT-H-14 --batch_size=32 --n_samples_save=0 \
                    --max_lr=3e-4 --mixup_pct=.33 --num_epochs=240 --ckpt_interval=5 --no-use_image_aug\
                    --prior --wandb_log --plot_umap"
    
  
    brain2caps_jupyter_args = "--data_path=/fsx/proj-medarc/fmri/natural-scenes-dataset \
                    --model_name=prior_blip_257x1408_subj01_bs24_heads22_no_img_aug_preproc\
                    --subj=1 --hidden --model_type=pretrain_opt6.7b --batch_size=24 --n_samples_save=0 \
                    --max_lr=3e-4 --mixup_pct=.33 --num_epochs=240 --ckpt_interval=5 --no-use_image_aug\
                    --prior --wandb_log --plot_umap --no-brain2clip"
    
    
    brain2clip_jupyter_args = brain2clip_jupyter_args.split()
    print("Brain2Clip:\n", brain2clip_jupyter_args)

    brain2caps_jupyter_args = brain2caps_jupyter_args.split()
    print("Brain2Caps:\n", brain2caps_jupyter_args)
    
    from IPython.display import clear_output # function to clear print outputs in cell
    %load_ext autoreload 
    %autoreload 2 # this allows you to change functions in models.py or utils.py and have this notebook automatically update with your revisions

In [ ]:
parser = argparse.ArgumentParser(description="Model Training Configuration")
parser.add_argument(
    "--brain2clip",action=argparse.BooleanOptionalAction,default=True,
    help="if True, train using image embeddings used in clip variant. If False, train using image embeddings from EVA-CLIP used in BLIP2 (brain2caps)",
)
parser.add_argument(
    "--neurovis",action=argparse.BooleanOptionalAction,default=True,
    help="If True, apply neurovis specific training loop modifications.",
)
parser.add_argument(
    "--model_name", type=str, default="testing",
    help="name of model, used for ckpt saving and wandb logging (if enabled)",
)
parser.add_argument(
    "--data_path", type=str, default="/fsx/brain-decoding/fmri/natural-scenes-dataset",
    help="Path to where NSD data is stored / where to download it to",
)
parser.add_argument(
    "--subj",type=int, default=1, choices=[1,2,5,7],
)
parser.add_argument(
    "--batch_size", type=int, default=32,
    help="Batch size can be increased by 10x if only training v2c and not diffusion prior",
)
parser.add_argument(
    "--hidden",action=argparse.BooleanOptionalAction,default=True,
    help="If True, CLIP embeddings will come from last hidden layer (e.g., 257 x 1280 - SD + IP Plus Adapter), rather than final layer",
)
parser.add_argument(
    # "--clip_variant",type=str,default="ViT-L/14",choices=["RN50", "ViT-L/14", "ViT-B/32", "RN50x64"],
    "--clip_variant",type=str,default="ViT-L/14",choices=["RN50", "ViT-L/14", "ViT-B/32", "RN50x64", "ViT-H-14"],
    help='OpenAI clip variant',
)
parser.add_argument(
    "--wandb_log",action=argparse.BooleanOptionalAction,default=False,
    help="whether to log to wandb",
)
parser.add_argument(
    "--resume_from_ckpt",action=argparse.BooleanOptionalAction,default=False,
    help="if not using wandb and want to resume from a ckpt",
)
parser.add_argument(
    "--wandb_project",type=str,default="stability",
    help="wandb project name",
)
parser.add_argument(
    "--mixup_pct",type=float,default=.33,
    help="proportion of way through training when to switch from BiMixCo to SoftCLIP",
)
parser.add_argument(
    "--norm_embs",action=argparse.BooleanOptionalAction,default=True,
    help="Do l2-norming of CLIP embeddings",
)
parser.add_argument(
    "--use_image_aug",action=argparse.BooleanOptionalAction,default=True,
    help="whether to use image augmentation",
)
parser.add_argument(
    "--num_epochs",type=int,default=240,
    help="number of epochs of training",
)
parser.add_argument(
    "--prior",action=argparse.BooleanOptionalAction,default=True,
    help="if False, will only use CLIP loss and ignore diffusion prior",
)
parser.add_argument(
    "--v2c",action=argparse.BooleanOptionalAction,default=True,
    help="if False, will only use diffusion prior loss",
)
parser.add_argument(
    "--plot_umap",action=argparse.BooleanOptionalAction,default=False,
    help="Plot UMAP plots alongside reconstructions",
)
parser.add_argument(
    "--lr_scheduler_type",type=str,default='cycle',choices=['cycle','linear'],
)
parser.add_argument(
    "--ckpt_saving",action=argparse.BooleanOptionalAction,default=True,
)
parser.add_argument(
    "--ckpt_interval",type=int,default=5,
    help="save backup ckpt and reconstruct every x epochs",
)
parser.add_argument(
    "--save_at_end",action=argparse.BooleanOptionalAction,default=False,
    help="if True, saves best.ckpt at end of training. if False and ckpt_saving==True, will save best.ckpt whenever epoch shows best validation score",
)
parser.add_argument(
    "--seed",type=int,default=42,
)
parser.add_argument(
    "--max_lr",type=float,default=3e-4,
)
parser.add_argument(
    "--n_samples_save",type=int,default=0,choices=[0,1],
    help="Number of reconstructions for monitoring progress, 0 will speed up training",
)
parser.add_argument(
    "--use_projector",action=argparse.BooleanOptionalAction,default=True,
    help="Additional MLP after the main MLP so model can separately learn a way to minimize NCE from prior loss (BYOL)",
)
parser.add_argument(
    "--vd_cache_dir", type=str, default='/fsx/proj-medarc/fmri/cache/models--shi-labs--versatile-diffusion/snapshots/2926f8e11ea526b562cd592b099fcf9c2985d0b7',
    help="Where is cached Versatile Diffusion model; if not cached will download to this path",
)
parser.add_argument(
    "--model_type",type=str,default="pretrain_opt6.7b",choices=["pretrain_opt2.7b", "pretrain_opt6.7b", "caption_coco_opt2.7b", "caption_coco_opt6.7b"],
    help='BLIP2 model type',
)


if utils.is_interactive():
    args = parser.parse_args(brain2clip_jupyter_args) if brain2clip else parser.parse_args(brain2caps_jupyter_args)
else:
    args = parser.parse_args()

# create global variables without the args prefix
for attribute_name in vars(args).keys():
    globals()[attribute_name] = getattr(args, attribute_name)
    
# need non-deterministic CuDNN for conv3D to work
utils.seed_everything(seed, cudnn_deterministic=False)

# change learning rate based on number of devices
max_lr *= accelerator.num_processes
    
# change batch size based on number of devices if using multi-gpu
# batch_size *= accelerator.num_processes

# change num_epochs based on number of devices if using multi-gpu
num_epochs *= accelerator.num_processes

In [ ]:
outdir = os.path.abspath(f'../train_logs/{model_name}')
print(outdir)
if not os.path.exists(outdir):
    os.makedirs(outdir,exist_ok=True)
if use_image_aug:
    import kornia
    from kornia.augmentation.container import AugmentationSequential
    img_augment = AugmentationSequential(
        kornia.augmentation.RandomResizedCrop((224,224), (0.6,1), p=0.3),
        kornia.augmentation.Resize((224, 224)),
        kornia.augmentation.RandomHorizontalFlip(p=0.5),
        kornia.augmentation.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1, p=0.3),
        kornia.augmentation.RandomGrayscale(p=0.3),
        data_keys=["input"],
    )
    print("Using image augmentation")

# Prep models and dataloaders

In [ ]:
print(f'Pulling NSD webdataset data for {subj}...')

train_url = "{" + f"{data_path}/train_subj0{subj}_" + "{0..17}.tar," + f"{data_path}/val_subj0{subj}_0.tar" + "}"
val_url = f"{data_path}/test_subj0{subj}_" + "{0..1}.tar"
print(train_url,"\n",val_url)
meta_url = f"{data_path}/metadata_subj0{subj}.json"

# num_train = (8559 + 300) * 3
num_train = 8559 + 300
num_val = 982

print('Prepping train and validation dataloaders...')
train_dl, val_dl, num_train, num_val = utils.get_dataloaders(
    batch_size,'images',
    num_devices=num_devices,
    num_workers=num_workers,
    train_url=train_url,
    val_url=val_url,
    meta_url=meta_url,
    num_train=num_train,
    num_val=num_val,
    val_batch_size=300,
    cache_dir=data_path, #"/tmp/wds-cache",
    seed=seed,
    voxels_key='nsdgeneral.npy',
    to_tuple=["voxels", "images", "coco"],
    local_rank=local_rank,
    world_size=world_size,
    subj=subj,
)

In [ ]:
print('Creating Clipper...')

if brain2clip:
    print(f"Using CLIP {clip_variant}. To replicate Neuro-Vis, use ViT-H-14.")
    clip_sizes = {"RN50": 1024, "ViT-L/14": 768, "ViT-B/32": 512, "ViT-H-14": 1024}
    clip_size = clip_sizes[clip_variant]
else:
    assert model_type == "pretrain_opt6.7b", "Other BLIP2 model types are not yet supported."
    clip_variant = model_type
    
if hidden:
    if brain2clip:
        if clip_variant == "ViT-H-14":
            print("Using hidden layer CLIP space (SD + IP Adapter Plus)")
            if norm_embs:
                print("WARNING: YOU WANT UN-NORMED EMBEDDINGS FOR IP ADAPTER PLUS!")

            clip_size=1280
            out_dim = 257 * clip_size
        else:
            clip_extractor = Clipper(clip_variant, device=device, hidden_state=True, norm_embs=norm_embs)
            out_dim = 257 * clip_size
    else:
        print("Using hidden layer CLIP space (BLIP2)")
        clip_size=1408
        out_dim = 257 * clip_size
    
else:    
    assert clip_variant == "ViT-H-14"
    print("Using final layer ViT-H-14 CLIP space (SD + IP Adapter)")
    
    if norm_embs:
        print("WARNING: YOU WANT UN-NORMED EMBEDDINGS FOR SD + IP ADAPTER!")
    clip_extractor = OpenClipper(clip_variant, device=device, hidden_state=False, norm_embs=norm_embs)
    out_dim = clip_size

print("img embs variant:", clip_variant)
print("out_dim:",out_dim)

In [ ]:
print('Creating voxel2clip...')
if subj == 1:
    num_voxels = 15724
elif subj == 2:
    num_voxels = 14278
elif subj == 3:
    num_voxels = 15226
elif subj == 4:
    num_voxels = 13153
elif subj == 5:
    num_voxels = 13039
elif subj == 6:
    num_voxels = 17907
elif subj == 7:
    num_voxels = 12682
elif subj == 8:
    num_voxels = 14386
voxel2clip_kwargs = dict(in_dim=num_voxels,out_dim=out_dim,clip_size=clip_size,use_projector=use_projector)
voxel2clip = BrainNetwork(**voxel2clip_kwargs)

print(f"number of voxels: {num_voxels}")

In [ ]:
# load from ckpt
voxel2clip_path = "None"
if voxel2clip_path!="None":
    checkpoint = torch.load(voxel2clip_path, map_location='cpu')
    voxel2clip.load_state_dict(checkpoint['model_state_dict'],strict=False)
    del checkpoint
    
print("params of voxel2clip:")
if local_rank==0:
    utils.count_params(voxel2clip)

In [ ]:
# setup prior network
out_dim = clip_size
depth = 6

dim_head = 64
heads = clip_size//64  #22

if hidden:
    guidance_scale = 7.5
    timesteps = 100
    num_tokens = 257
  
    
    prior_network = PriorNetwork(
        dim=out_dim,
        depth=depth,
        dim_head=dim_head,
        heads=heads,
        causal=False,
        num_tokens = num_tokens,
        learned_query_mode="pos_emb"
    )
    
    print("num_tokens: ", num_tokens)
    print("hidden prior_network loaded")

    
    diffusion_prior = BrainDiffusionPriorV2(
        net=prior_network,
        image_embed_dim=out_dim,
        condition_on_text_encodings=False,
        timesteps=timesteps,
        cond_drop_prob=0.2,
        image_embed_scale=None,
        voxel2clip=voxel2clip,
    ).to(device)
    
    
else:
    # guidance_scale = 7.5
    # timesteps = 1000
    # diffusion_prior = BrainDiffusionPriorOld.from_pretrained(
    #     # kwargs for DiffusionPriorNetwork
    #     dict(),
    #     # kwargs for DiffusionNetwork
    #     dict(
    #         condition_on_text_encodings=False,
    #         timesteps=timesteps,
    #         voxel2clip=voxel2clip,
    #     ),
    #     voxel2clip_path=None,
    #     ckpt_dir='/notebooks/brain-decoding/mindeye/checkpoints',
    # )
    
    guidance_scale = 7.5
    timesteps = 100
    prior_network = StableDiffusionPriorNetwork(
            dim=out_dim,
            depth=depth,
            dim_head=dim_head,
            heads=heads,
            causal=False,
            num_tokens = 1,
            learned_query_mode="pos_emb"
        ).to(device)
    print("prior_network loaded")

    # custom version that can fix seeds
    diffusion_prior = BrainDiffusionPrior(
        net=prior_network,
        image_embed_dim=out_dim,
        condition_on_text_encodings=False,
        timesteps=timesteps,
        cond_drop_prob=0.2,
        image_embed_scale=None,
        voxel2clip=voxel2clip,
    ).to(device)

In [ ]:
prior

In [ ]:
if not prior:
    diffusion_prior = diffusion_prior.requires_grad_(False)
    diffusion_prior.voxel2clip.requires_grad_(True)

print("params of diffusion prior:")
if local_rank==0:
    utils.count_params(diffusion_prior)

no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
opt_grouped_parameters = [
    {'params': [p for n, p in diffusion_prior.net.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 1e-2},
    {'params': [p for n, p in diffusion_prior.net.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    {'params': [p for n, p in diffusion_prior.voxel2clip.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 1e-2},
    {'params': [p for n, p in diffusion_prior.voxel2clip.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = torch.optim.AdamW(opt_grouped_parameters, lr=max_lr)

global_batch_size = batch_size * num_devices
if lr_scheduler_type == 'linear':
    lr_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        total_iters=int(num_epochs*(num_train//global_batch_size)),
        last_epoch=-1
    )
elif lr_scheduler_type == 'cycle':
    total_steps=int(num_epochs*(num_train//global_batch_size))
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=max_lr,
        total_steps=total_steps,
        final_div_factor=1000,
        last_epoch=-1, pct_start=2/num_epochs
    )

In [ ]:
if plot_umap:
    import umap

assert n_samples_save == 0, "Number of recons > 0 is not yet implemented for SD + IP Adapter Plus here."
if n_samples_save > 0 and hidden:
    # print('Creating versatile diffusion reconstruction pipeline...')
    # from diffusers import VersatileDiffusionDualGuidedPipeline, UniPCMultistepScheduler
    # from diffusers.models import DualTransformer2DModel
    # try:
    #     vd_pipe =  VersatileDiffusionDualGuidedPipeline.from_pretrained(vd_cache_dir).to('cpu')
    # except:
    #     print("Downloading Versatile Diffusion to", vd_cache_dir)
    #     vd_pipe =  VersatileDiffusionDualGuidedPipeline.from_pretrained(
    #             "shi-labs/versatile-diffusion",
    #             cache_dir = vd_cache_dir).to('cpu')
    # vd_pipe.image_unet.eval()
    # vd_pipe.vae.eval()
    # vd_pipe.image_unet.requires_grad_(False)
    # vd_pipe.vae.requires_grad_(False)

    # vd_pipe.scheduler = UniPCMultistepScheduler.from_pretrained(vd_cache_dir, subfolder="scheduler")
    # num_inference_steps = 20

    # # Set weighting of Dual-Guidance 
    # text_image_ratio = .0 # .5 means equally weight text and image, 0 means use only image
    # for name, module in vd_pipe.image_unet.named_modules():
    #     if isinstance(module, DualTransformer2DModel):
    #         module.mix_ratio = text_image_ratio
    #         for i, type in enumerate(("text", "image")):
    #             if type == "text":
    #                 module.condition_lengths[i] = 77
    #                 module.transformer_index_for_condition[i] = 1  # use the second (text) transformer
    #             else:
    #                 module.condition_lengths[i] = 257
    #                 module.transformer_index_for_condition[i] = 0  # use the first (image) transformer
                    
    # unet = vd_pipe.image_unet
    # vae = vd_pipe.vae
    # noise_scheduler = vd_pipe.scheduler
    pass

elif n_samples_save > 0:
#     print('Creating SD image variations reconstruction pipeline...')
#     from diffusers import AutoencoderKL, UNet2DConditionModel, UniPCMultistepScheduler

#     sd_cache_dir = '/fsx/home/.cache/huggingface/diffusers/models--lambdalabs--sd-image-variations-diffusers/snapshots/a2a13984e57db80adcc9e3f85d568dcccb9b29fc'
#     unet = UNet2DConditionModel.from_pretrained(sd_cache_dir,subfolder="unet").to(device)

#     unet.eval() # dont want to train model
#     unet.requires_grad_(False) # dont need to calculate gradients

#     vae = AutoencoderKL.from_pretrained(sd_cache_dir,subfolder="vae").to(device)
#     vae.eval()
#     vae.requires_grad_(False)

#     noise_scheduler = UniPCMultistepScheduler.from_pretrained(sd_cache_dir, subfolder="scheduler")
#     num_inference_steps = 20
    pass

In [ ]:
def save_ckpt(tag):    
    ckpt_path = outdir+f'/{tag}.pth'
    print(f'saving {ckpt_path}',flush=True)
    unwrapped_model = accelerator.unwrap_model(diffusion_prior)
    try:
        torch.save({
            'epoch': epoch,
            'model_state_dict': unwrapped_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'lr_scheduler': lr_scheduler.state_dict(),
            'train_losses': losses,
            'val_losses': val_losses,
            'lrs': lrs,
            }, ckpt_path)
    except:
        print("Couldn't save... moving on to prevent crashing.")
    del unwrapped_model
        
print("\nDone with model preparations!")

# Weights and Biases 

In [ ]:
# params for wandb
if local_rank==0 and wandb_log: # only use main process for wandb logging
    import wandb
    
    # wandb_project = 'stability'
    wandb_project = 'brain-decoding'
    wandb_run = model_name
    # wandb_run = wandb_run_name
    wandb_notes = ''
    
    print(f"wandb {wandb_project} run {wandb_run}")
    # wandb.login(host='https://stability.wandb.io')#, relogin=True)
    wandb.login()
    wandb_config = {
      "model_name": model_name,
      "clip_variant": clip_variant,
      "batch_size": batch_size,
      "num_epochs": num_epochs,
      "use_image_aug": use_image_aug,
      "max_lr": max_lr,
      "lr_scheduler_type": lr_scheduler_type,
      "mixup_pct": mixup_pct,
      "num_train": num_train,
      "num_val": num_val,
      "seed": seed,
      "distributed": distributed,
      "num_devices": num_devices,
      "world_size": world_size,
      "train_url": train_url,
      "val_url": val_url,
    }
    print("wandb_config:\n",wandb_config)
    if True: # wandb_auto_resume
        print("wandb_id:",model_name)
        wandb.init(
            id = model_name,
            project=wandb_project,
            name=wandb_run,
            config=wandb_config,
            notes=wandb_notes,
            resume="allow",
        )
    else:
        wandb.init(
            project=wandb_project,
            name=wandb_run,
            config=wandb_config,
            notes=wandb_notes,
        )
else:
    wandb_log = False

In [ ]:
epoch = 0
losses, val_losses, lrs = [], [], []
nce_losses, val_nce_losses = [], []
sim_losses, val_sim_losses = [], []
best_val_loss = 1e9
soft_loss_temps = utils.cosine_anneal(0.004, 0.0075, num_epochs - int(mixup_pct * num_epochs))
if hidden:
    prior_mult = 30
else:
    prior_mult = .03
val_voxel0 = val_image0 = None

# Optionally resume from checkpoint #
if resume_from_ckpt:
    print("\n---resuming from last.pth ckpt---\n")
    try:
        checkpoint = torch.load(outdir+'/last.pth', map_location='cpu')
    except:
        print('last.pth failed... trying last_backup.pth')
        checkpoint = torch.load(outdir+'/last_backup.pth', map_location='cpu')
    epoch = checkpoint['epoch']
    print("Epoch",epoch)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
    diffusion_prior.load_state_dict(checkpoint['model_state_dict'])
    del checkpoint
elif wandb_log:
    if wandb.run.resumed:
        print("\n---resuming from last.pth ckpt---\n")
        try:
            checkpoint = torch.load(outdir+'/last.pth', map_location='cpu')
        except:
            print('last.pth failed... trying last_backup.pth')
            checkpoint = torch.load(outdir+'/last_backup.pth', map_location='cpu')
        epoch = checkpoint['epoch']
        print("Epoch",epoch)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
        diffusion_prior.load_state_dict(checkpoint['model_state_dict'])
        del checkpoint
torch.cuda.empty_cache()

In [ ]:
diffusion_prior, optimizer, train_dl, val_dl, lr_scheduler = accelerator.prepare(
diffusion_prior, optimizer, train_dl, val_dl, lr_scheduler
)

# Load Image Encoder

In [4]:
# TO DO: move to models.py
if brain2clip == False:
    import logging
    from packaging import version
    from lavis.common.registry import registry
    from lavis.models import load_model_and_preprocess
    from lavis.models.blip2_models.blip2 import Blip2Base, disabled_train
    # from lavis.models.blip2_models.modeling_opt import OPTForCausalLM, OPTConfig

    from transformers import AutoTokenizer, OPTForCausalLM, OPTConfig
    import transformers #must be >= 4.27.0

    @registry.register_model("blip2_opt_v2")
    class Blip2OPT(Blip2Base):
        """
        BLIP2 OPT model.
        Supported model types:
            - pretrained_opt2.7b: pretrained model with OPT2.7b
            - pretrained_opt6.7b: pretrained model with OPT6.7b
            - caption_coco_opt2.7b: fintuned image captioning model with OPT2.7b
            - caption_coco_opt6.7b: fintuned image captioning model with OPT6.7b
        Usage:
            >>> from lavis.models import load_model
            >>> model = load_model("blip2_opt", "caption_coco_opt2.7b")
        """

        PRETRAINED_MODEL_CONFIG_DICT = {
            "pretrain_opt2.7b": "configs/models/blip2/blip2_pretrain_opt2.7b.yaml",
            "pretrain_opt6.7b": "configs/models/blip2/blip2_pretrain_opt6.7b.yaml",
            "caption_coco_opt2.7b": "configs/models/blip2/blip2_caption_opt2.7b.yaml",
            "caption_coco_opt6.7b": "configs/models/blip2/blip2_caption_opt6.7b.yaml",
        }

        def __init__(
            self,
            vit_model="eva_clip_g",
            img_size=224,
            drop_path_rate=0,
            use_grad_checkpoint=False,
            vit_precision="fp16",
            freeze_vit=True,
            num_query_token=32,
            opt_model="facebook/opt-2.7b",
            prompt="",
            max_txt_len=32,
            apply_lemmatizer=False,
        ):
            """
            apply_lemmatizer: when set to True, postprocess predict_answers() result with lemmas.
            """
            super().__init__()
            transformers_version = version.parse(transformers.__version__)
            assert transformers_version >= version.parse("4.27"), "BLIP-2 OPT requires transformers>=4.27"

            self.tokenizer = self.init_tokenizer()

            self.visual_encoder, self.ln_vision = self.init_vision_encoder(
                vit_model, img_size, drop_path_rate, use_grad_checkpoint, vit_precision
            )
            if freeze_vit:
                for name, param in self.visual_encoder.named_parameters():
                    param.requires_grad = False
                self.visual_encoder = self.visual_encoder.eval()
                self.visual_encoder.train = disabled_train
                logging.info("freeze vision encoder")

            self.Qformer, self.query_tokens = self.init_Qformer(
                num_query_token, self.visual_encoder.num_features
            )
            self.Qformer.cls = None
            self.Qformer.bert.embeddings.word_embeddings = None
            self.Qformer.bert.embeddings.position_embeddings = None
            for layer in self.Qformer.bert.encoder.layer:
                layer.output = None
                layer.intermediate = None

            self.opt_tokenizer = AutoTokenizer.from_pretrained(opt_model, use_fast=False)
            self.opt_model = OPTForCausalLM.from_pretrained(
                opt_model, torch_dtype=torch.float16
            )
            for name, param in self.opt_model.named_parameters():
                param.requires_grad = False
            self.eos_token_id = self.opt_tokenizer(
                "\n", add_special_tokens=False
            ).input_ids[0]

            self.opt_proj = nn.Linear(
                self.Qformer.config.hidden_size, self.opt_model.config.hidden_size
            )

            self.max_txt_len = max_txt_len
            self.prompt = prompt
            prompt_tokens = self.opt_tokenizer(self.prompt, return_tensors="pt")
            self.prompt_length = prompt_tokens.attention_mask.sum(1)

            self._apply_lemmatizer = apply_lemmatizer
            self._lemmatizer = None

        def forward(self, samples):
            image = samples["image"]
            with self.maybe_autocast():
                image_embeds = self.ln_vision(self.visual_encoder(image))
            image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(
                image.device
            )

            query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
            query_output = self.Qformer.bert(
                query_embeds=query_tokens,
                encoder_hidden_states=image_embeds,
                encoder_attention_mask=image_atts,
                return_dict=True,
            )

            inputs_opt = self.opt_proj(query_output.last_hidden_state)
            atts_opt = torch.ones(inputs_opt.size()[:-1], dtype=torch.long).to(image.device)

            self.opt_tokenizer.padding_side = "right"

            text = [t + "\n" for t in samples["text_input"]]

            opt_tokens = self.opt_tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                truncation=True,
                max_length=self.max_txt_len,
            ).to(image.device)

            targets = opt_tokens.input_ids.masked_fill(
                opt_tokens.input_ids == self.opt_tokenizer.pad_token_id, -100
            )
            if self.prompt:
                targets[:, : self.prompt_length] = -100  # do not apply loss to the prompt

            empty_targets = (
                torch.ones(atts_opt.size(), dtype=torch.long).to(image.device).fill_(-100)
            )
            targets = torch.cat([empty_targets, targets], dim=1)

            inputs_embeds = self.opt_model.model.decoder.embed_tokens(opt_tokens.input_ids)
            inputs_embeds = torch.cat([inputs_opt, inputs_embeds], dim=1)
            attention_mask = torch.cat([atts_opt, opt_tokens.attention_mask], dim=1)

            with self.maybe_autocast():
                outputs = self.opt_model(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    return_dict=True,
                    labels=targets,
                )
            loss = outputs.loss

            return {"loss": loss}

        @torch.no_grad()
        def generate(
            self,
            samples,
            image_embeds,
            image_size,
            device,
            use_nucleus_sampling=False,
            num_beams=5,
            max_length=30,
            min_length=1,
            top_p=0.9,
            repetition_penalty=1.0,
            length_penalty=1.0,
            num_captions=1,
            temperature=1,
        ):
            """
            Args:
                samples (dict): A dictionary containing the following keys:
                    - image (torch.Tensor): A tensor of shape (batch_size, 3, H, W)
                use_nucleus_sampling (bool): Whether to use nucleus sampling. If False, use top-k sampling.
                num_beams (int): Number of beams for beam search. 1 means no beam search.
                max_length (int): The maximum length of the sequence to be generated.
                min_length (int): The minimum length of the sequence to be generated.
                top_p (float): The cumulative probability for nucleus sampling.
                repetition_penalty (float): The parameter for repetition penalty. 1.0 means no penalty.
                num_captions (int): Number of captions to be generated for each image.
            Returns:
                captions (list): A list of strings of length batch_size * num_captions.
            """
            # image = samples["image"]
            with self.maybe_autocast():
                # image_embeds = self.ln_vision(self.visual_encoder(image))
                image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(
                    device
                )

                # breakpoint()

                query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
                query_output = self.Qformer.bert(
                    query_embeds=query_tokens,
                    encoder_hidden_states=image_embeds,
                    encoder_attention_mask=image_atts,
                    return_dict=True,
                )

                inputs_opt = self.opt_proj(query_output.last_hidden_state)
                atts_opt = torch.ones(inputs_opt.size()[:-1], dtype=torch.long).to(
                    device
                )

                # breakpoint()

                if "prompt" in samples.keys():
                    prompt = samples["prompt"]
                else:
                    prompt = self.prompt

                # prompt = [prompt] * image.size(0)
                prompt = [prompt] * image_size

                opt_tokens = self.opt_tokenizer(
                    prompt,
                    return_tensors="pt",
                    padding="longest",
                    truncation=True,
                    max_length=self.max_txt_len,
                ).to(device)
                attention_mask = torch.cat([atts_opt, opt_tokens.attention_mask], dim=1)

                # breakpoint()

                # new version for transformers>=4.27
                inputs_embeds = self.opt_model.get_input_embeddings()(opt_tokens.input_ids)
                inputs_embeds = torch.cat([inputs_opt,inputs_embeds],dim=1)

                outputs = self.opt_model.generate(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    do_sample=use_nucleus_sampling,
                    top_p=top_p,
                    temperature=temperature,
                    num_beams=num_beams,
                    max_length=max_length,
                    min_length=min_length,
                    eos_token_id=self.eos_token_id,
                    repetition_penalty=repetition_penalty,
                    length_penalty=length_penalty,
                    num_return_sequences=num_captions,
                )
                output_text = self.opt_tokenizer.batch_decode(
                    outputs, skip_special_tokens=True
                )

                # breakpoint()
                # previous version for transformers<4.27
                # if use_nucleus_sampling:
                #     query_embeds = inputs_opt.repeat_interleave(num_captions, dim=0)
                #     num_beams = 1
                # else:
                #     query_embeds = inputs_opt.repeat_interleave(num_beams, dim=0)

                # outputs = self.opt_model.generate(
                #     input_ids=input_ids,
                #     query_embeds=query_embeds,
                #     attention_mask=attention_mask,
                #     do_sample=use_nucleus_sampling,
                #     top_p=top_p,
                #     temperature=temperature,
                #     num_beams=num_beams,
                #     max_new_tokens=max_length,
                #     min_length=min_length,
                #     eos_token_id=self.eos_token_id,
                #     repetition_penalty=repetition_penalty,
                #     length_penalty=length_penalty,
                #     num_return_sequences=num_captions,
                # )

                # prompt_length = opt_tokens.input_ids.shape[1]
                # output_text = self.opt_tokenizer.batch_decode(
                #     outputs[:, prompt_length:], skip_special_tokens=True
                # )

                output_text = [text.strip() for text in output_text]
                return output_text

        def predict_answers(
            self,
            samples,
            num_beams=5,
            inference_method="generate",
            max_len=10,
            min_len=1,
            num_ans_candidates=128,
            answer_list=None,
            prompt="",
            length_penalty=0,
            **kwargs
        ):
            image = samples["image"]
            with self.maybe_autocast():
                image_embeds = self.ln_vision(self.visual_encoder(image))
                image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(
                    image.device
                )

                query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
                query_output = self.Qformer.bert(
                    query_embeds=query_tokens,
                    encoder_hidden_states=image_embeds,
                    encoder_attention_mask=image_atts,
                    return_dict=True,
                )

                inputs_opt = self.opt_proj(query_output.last_hidden_state)
                atts_opt = torch.ones(inputs_opt.size()[:-1], dtype=torch.long).to(
                    image.device
                )

                if isinstance(samples["text_input"], str):
                    samples["text_input"] = [samples["text_input"]]
                if prompt:
                    text_input = [prompt.format(question) for question in samples["text_input"]]
                else:
                    text_input = samples["text_input"]

                self.opt_tokenizer.padding_side = "left"
                opt_tokens = self.opt_tokenizer(
                    text_input,
                    return_tensors="pt",
                    padding="longest",
                    truncation=True,
                    max_length=self.max_txt_len,
                ).to(image.device)

                attention_mask = torch.cat([atts_opt, opt_tokens.attention_mask], dim=1)

                # require transformers>=4.27
                inputs_embeds = self.opt_model.get_input_embeddings()(opt_tokens.input_ids)
                inputs_embeds = torch.cat([inputs_opt,inputs_embeds],dim=1)

                outputs = self.opt_model.generate(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    do_sample=False,
                    num_beams=num_beams,
                    max_new_tokens=max_len,
                    min_length=min_len,
                    eos_token_id=self.eos_token_id,
                    length_penalty=length_penalty,
                )
                output_text = self.opt_tokenizer.batch_decode(
                    outputs, skip_special_tokens=True
                )
                output_text = [text.strip() for text in output_text]
            if self._apply_lemmatizer or ("apply_lemmatizer" in samples.keys() and samples["apply_lemmatizer"]):
                output_text = self._lemmatize(output_text)

            return output_text

        def _lemmatize(self, answers):
            def apply(answer):
                doc = self.lemmatizer(answer)

                words = []
                for token in doc:
                    if token.pos_ in ["NOUN", "VERB"]:
                        words.append(token.lemma_)
                    else:
                        words.append(token.text)
                answer = " ".join(words)

                return answer

            return [apply(answer) for answer in answers]

        @property
        def lemmatizer(self):
            if self._lemmatizer is None:
                try:
                    import spacy

                    self._lemmatizer = spacy.load("en_core_web_sm")
                except ImportError:
                    logging.error(
                        """
                        Please install spacy and en_core_web_sm model to apply lemmatization.
                        python -m spacy download en_core_web_sm
                        OR
                        import spacy.cli
                        spacy.cli.download("en_core_web_sm")
                        """
                    )
                    exit(1)

            return self._lemmatizer

        @classmethod
        def from_config(cls, cfg):
            vit_model = cfg.get("vit_model", "eva_clip_g")
            img_size = cfg.get("image_size")
            num_query_token = cfg.get("num_query_token")
            opt_model = cfg.get("opt_model")

            drop_path_rate = cfg.get("drop_path_rate", 0)
            use_grad_checkpoint = cfg.get("use_grad_checkpoint", False)
            vit_precision = cfg.get("vit_precision", "fp16")
            freeze_vit = cfg.get("freeze_vit", True)

            prompt = cfg.get("prompt", "")
            max_txt_len = cfg.get("max_txt_len", 32)

            apply_lemmatizer = cfg.get("apply_lemmatizer", False)

            model = cls(
                vit_model=vit_model,
                img_size=img_size,
                drop_path_rate=drop_path_rate,
                use_grad_checkpoint=use_grad_checkpoint,
                vit_precision=vit_precision,
                freeze_vit=freeze_vit,
                num_query_token=num_query_token,
                opt_model=opt_model,
                prompt=prompt,
                max_txt_len=max_txt_len,
                apply_lemmatizer=apply_lemmatizer,
            )
            model.load_checkpoint_from_config(cfg)

            return model

In [ ]:
if brain2clip and clip_variant == "ViT-H-14":
    print("Loading CLIP ViT-H image encoder...")
    
    from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
    
    # overwrite preprocess to accept torch inputs instead of PIL Image
    preprocess = transforms.Compose([
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=None),
            transforms.CenterCrop(224),
            transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
    ])
    
    image_encoder = CLIPVisionModelWithProjection.from_pretrained('laion/CLIP-ViT-H-14-laion2B-s32B-b79K').to(
        device, dtype=torch.float32
    )

    image_encoder.eval() # dont want to train model
    for param in image_encoder.parameters():
        param.requires_grad = False # dont need to calculate gradients

    # clip_image_processor = CLIPImageProcessor()
  
elif brain2clip == False:
    
    print("Loading BLIP2 image encoder...")
    
    preprocess = transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC, antialias=None),
        # transforms.CenterCrop(364),
        transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
    ])
    
    model, vis_processors, _= load_model_and_preprocess(
        name="blip2_opt_v2", model_type="pretrain_opt6.7b", is_eval=True, device=device
    )

# Train

In [ ]:
print(f"{model_name} starting with epoch {epoch} / {num_epochs}")
progress_bar = tqdm(range(epoch,num_epochs), ncols=1200, disable=(local_rank!=0))

for epoch in progress_bar:
    diffusion_prior.train()

    sims_base = 0.
    val_sims_base = 0.
    fwd_percent_correct = 0.
    bwd_percent_correct = 0.
    val_fwd_percent_correct = 0.
    val_bwd_percent_correct = 0.
    loss_nce_sum = 0.
    loss_prior_sum = 0.
    val_loss_nce_sum = 0.
    val_loss_prior_sum = 0.

    for train_i, (voxel, image, coco) in enumerate(train_dl):
        # break
        with torch.cuda.amp.autocast():
            optimizer.zero_grad()

            repeat_index = train_i % 3
            
            if use_image_aug:
                image = img_augment(image)
                # plt.imshow(utils.torch_to_Image(image))
                # plt.show()

            voxel = voxel[:,repeat_index].float()

            if epoch < int(mixup_pct * num_epochs):
                voxel, perm, betas, select = utils.mixco(voxel)
            
            if neurovis:            
                clip_proc = preprocess(image)
                clip_proc = clip_proc.to(device, dtype=torch.float16)
                
                if v2clip:
                    clip_image_embeds = image_encoder(clip_proc, output_hidden_states=True).hidden_states[-2]
                else:
                    with torch.no_grad():
                        with model.maybe_autocast():
                            clip_image_embeds = model.ln_vision(model.visual_encoder(clip_proc))
                            
                clip_target = clip_image_embeds.float()
                assert not torch.any(torch.isnan(clip_target))
                
            else:
                clip_target = clip_extractor.embed_image(image).float()   

            clip_voxels, clip_voxels_proj = diffusion_prior.module.voxel2clip(voxel) if distributed else diffusion_prior.voxel2clip(voxel)
            
            if hidden or neurovis:
                clip_voxels = clip_voxels.view(len(voxel),-1,clip_size)
            
            
            if neurovis:
                clip_target = clip_target.view(len(clip_target),-1,clip_size)
            
            if prior:
                loss_prior, aligned_clip_voxels = diffusion_prior(text_embed=clip_voxels, image_embed=clip_target)
                aligned_clip_voxels /= diffusion_prior.module.image_embed_scale if distributed else diffusion_prior.image_embed_scale
                
                
            else:
                aligned_clip_voxels = clip_voxels

            clip_voxels_norm = nn.functional.normalize(clip_voxels_proj.flatten(1), dim=-1)
            clip_target_norm = nn.functional.normalize(clip_target.flatten(1), dim=-1)
            

            if epoch < int(mixup_pct * num_epochs):
                loss_nce = utils.mixco_nce(
                    clip_voxels_norm,
                    clip_target_norm,
                    temp=.006, 
                    perm=perm, betas=betas, select=select)
            else:
                epoch_temp = soft_loss_temps[epoch-int(mixup_pct*num_epochs)]
                loss_nce = utils.soft_clip_loss(
                    clip_voxels_norm,
                    clip_target_norm,
                    temp=epoch_temp)
                
            if prior and v2c:
                loss_nce_sum += loss_nce.item()
                loss_prior_sum += loss_prior.item()
                loss = loss_nce + (prior_mult * loss_prior)
            elif v2c:
                loss_nce_sum += loss_nce.item()
                loss = loss_nce
            elif prior:
                loss_prior_sum += loss_prior.item()
                loss = prior_mult * loss_prior
            utils.check_loss(loss)
            
            accelerator.backward(loss)
            optimizer.step()

            losses.append(loss.item())
            lrs.append(optimizer.param_groups[0]['lr'])

            # gather batches across multi-gpu if there's multiple
            # clip_voxel_gather = accelerator.gather(clip_voxels_norm.view(len(voxel),-1).contiguous())
            # clip_target_gather = accelerator.gather(clip_target_norm.view(len(voxel),-1).contiguous())

            sims_base += nn.functional.cosine_similarity(clip_target_norm,clip_voxels_norm).mean().item()

            # forward and backward top 1 accuracy        
            labels = torch.arange(len(clip_target_norm)).to(device) 
            fwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_voxels_norm,clip_target_norm), labels, k=1)
            bwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_target_norm, clip_voxels_norm), labels, k=1)

            if lr_scheduler_type is not None:
                lr_scheduler.step()
            

    diffusion_prior.eval()
    for val_i, (voxel, image, coco) in enumerate(val_dl): 
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                # repeat_index = val_i % 3

                # voxel = voxel[:,repeat_index].float()
                voxel = torch.mean(voxel,axis=1).float()
                
                if use_image_aug:
                    image = img_augment(image)

                if val_image0 is None:
                    val_image0 = image.detach().clone()
                    val_voxel0 = voxel.detach().clone()

                if neurovis:        
                    clip_proc = preprocess(image)
                    clip_proc = clip_proc.to(device, dtype=torch.float16)

                    if v2clip:
                        clip_image_embeds = image_encoder(clip_proc, output_hidden_states=True).hidden_states[-2]
                    else:
                        with torch.no_grad():
                            with model.maybe_autocast():
                                clip_image_embeds = model.ln_vision(model.visual_encoder(clip_proc))

                    clip_target = clip_image_embeds.float()
                    assert not torch.any(torch.isnan(clip_target))
                
                else:
                    clip_target = clip_extractor.embed_image(image).float()

                clip_voxels, clip_voxels_proj = diffusion_prior.module.voxel2clip(voxel) if distributed else diffusion_prior.voxel2clip(voxel)
                
                if hidden or neurovis:
                    clip_voxels = clip_voxels.view(len(voxel),-1,clip_size)
                
                if neurovis:
                    clip_target = clip_target.view(len(clip_target),-1,clip_size)
                
                if prior:
                    val_loss_prior, aligned_clip_voxels = diffusion_prior(text_embed=clip_voxels, image_embed=clip_target)
                    aligned_clip_voxels /= diffusion_prior.module.image_embed_scale if distributed else diffusion_prior.image_embed_scale
                else:
                    aligned_clip_voxels = clip_voxels

                clip_voxels_norm = nn.functional.normalize(clip_voxels_proj.flatten(1), dim=-1)
                clip_target_norm = nn.functional.normalize(clip_target.flatten(1), dim=-1)

                if epoch < int(mixup_pct * num_epochs):
                    val_loss_nce = utils.mixco_nce(
                        clip_voxels_norm,
                        clip_target_norm,
                        temp=.006, 
                        perm=None, betas=None, select=None)
                else:
                    val_loss_nce = utils.soft_clip_loss(
                        clip_voxels_norm,
                        clip_target_norm,
                        temp=epoch_temp)

                if prior and v2c:
                    val_loss_nce_sum += val_loss_nce.item()
                    val_loss_prior_sum += val_loss_prior.item()
                    val_loss = val_loss_nce + (prior_mult * val_loss_prior)
                elif v2c:
                    val_loss_nce_sum += val_loss_nce.item()
                    val_loss = val_loss_nce
                elif prior:
                    val_loss_prior_sum += val_loss_prior.item()
                    val_loss = prior_mult * val_loss_prior
                utils.check_loss(val_loss)
                
                val_losses.append(val_loss.item())

                # clip_voxel_gather = accelerator.gather(clip_voxels_norm.view(len(voxel),-1).contiguous())
                # clip_target_gather = accelerator.gather(clip_target_norm.view(len(voxel),-1).contiguous())

                val_sims_base += nn.functional.cosine_similarity(clip_target_norm,clip_voxels_norm).mean().item()
                
                labels = torch.arange(len(clip_target_norm)).to(device)
                val_fwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_voxels_norm,clip_target_norm), labels, k=1)
                val_bwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_target_norm, clip_voxels_norm), labels, k=1)

    if local_rank==0:        
        if (not save_at_end and ckpt_saving) or (save_at_end and epoch == num_epochs - 1):
            # save best model
            val_loss = np.mean(val_losses[-(val_i+1):])
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_ckpt('best')
            else:
                print(f'not best - val_loss: {val_loss:.3f}, best_val_loss: {best_val_loss:.3f}')
                
        if utils.is_interactive():
            clear_output(wait=True)
            
        logs = {"train/loss": np.mean(losses[-(train_i+1):]),
            "val/loss": np.mean(val_losses[-(val_i+1):]),
            "train/lr": lrs[-1],
            "train/num_steps": len(losses),
            "val/num_steps": len(val_losses),
            "train/cosine_sim_base": sims_base / (train_i + 1),
            "val/cosine_sim_base": val_sims_base / (val_i + 1),
            "train/fwd_pct_correct": fwd_percent_correct / (train_i + 1),
            "train/bwd_pct_correct": bwd_percent_correct / (train_i + 1),
            "val/val_fwd_pct_correct": val_fwd_percent_correct / (val_i + 1),
            "val/val_bwd_pct_correct": val_bwd_percent_correct / (val_i + 1),
            "train/loss_nce": loss_nce_sum / (train_i + 1),
            "train/loss_prior": loss_prior_sum / (train_i + 1),
            "val/loss_nce": val_loss_nce_sum / (val_i + 1),
            "val/loss_prior": val_loss_prior_sum / (val_i + 1)}
        progress_bar.set_postfix(**logs)

        # Save model checkpoint and reconstruct
        save_ckpt(f'last')
        if epoch % ckpt_interval == 0:
            save_ckpt(f'last_backup')
            if n_samples_save > 0:
                del clip_voxels, clip_voxels_proj, image, voxel # free up some memory
                print('reconstructing...')
                with torch.no_grad():
                    if hidden:
                        vd_pipe = vd_pipe.to(device)
                    grid, _, _, _ = utils.reconstruction(
                        val_image0, val_voxel0,
                        clip_extractor, unet, vae, noise_scheduler,
                        diffusion_priors = diffusion_prior.module if distributed else diffusion_prior,
                        num_inference_steps = num_inference_steps,
                        n_samples_save = 1,
                        guidance_scale = guidance_scale,
                        timesteps_prior = timesteps,
                        seed = seed,
                        retrieve = False,
                        plotting = True,
                        img_variations = not hidden,
                        verbose=False,
                    )
                if wandb_log:
                    logs[f"val/recons"] = wandb.Image(grid, caption=f"epoch{epoch:03d}")
                    plt.close()
                else:
                    grid.savefig(os.path.join(outdir, f'samples-val-epoch{epoch:03d}.png'))
                    plt.show()
                if hidden:
                    vd_pipe = vd_pipe.to('cpu')
                
            if plot_umap:
                print('umap plotting...')
                combined = np.concatenate((clip_target.flatten(1).detach().cpu().numpy(),
                                           aligned_clip_voxels.flatten(1).detach().cpu().numpy()),axis=0)
                reducer = umap.UMAP(random_state=42)
                embedding = reducer.fit_transform(combined)

                colors=np.array([[0,0,1,.5] for i in range(len(clip_target))])
                colors=np.concatenate((colors, np.array([[0,1,0,.5] for i in range(len(aligned_clip_voxels))])))

                fig = plt.figure(figsize=(5,5))
                plt.scatter(
                    embedding[:, 0],
                    embedding[:, 1],
                    c=colors)
                if wandb_log:
                    logs[f"val/umap"] = wandb.Image(fig, caption=f"epoch{epoch:03d}")
                    plt.close()
                else:
                    plt.savefig(os.path.join(outdir, f'umap-val-epoch{epoch:03d}.png'))
                    plt.show()
        
        if wandb_log: wandb.log(logs)
        
    # wait for other GPUs to catch up if needed
    accelerator.wait_for_everyone()

print("\n===Finished!===\n")
if not utils.is_interactive():
    sys.exit(0)